In [ ]:
import numpy as np

from md_Helpers import cavitation
from md_Helpers import eos_sweep

kT_values = [0.700, 0.750, 0.800, 0.850]
n_fcc_cells_values = [10, 12, 15, 17, 20, 22, 25, 27, 30]
radii = [2.0, 3.0, 4.0, 5.0]

source_nsteps = 1_000_000
evolve_nsteps = 100_000
source_seed = 1
evolve_seed = 1

# Build/load EOS pressure-window sweep for n_cells=30.
eos_summary = eos_sweep.run_eos_pressure_window_sweep(
    n_fcc_cells=30,
    kT_start=min(kT_values),
    kT_stop=max(kT_values),
    kT_step=0.050,
    initial_rho=0.710,
    rho_step=0.005,
    nsteps=source_nsteps,
    seed=source_seed,
    overwrite=False,
)

pressure_col = "pressure_mean_last100"

for n_fcc_cells in n_fcc_cells_values:
    for kT in kT_values:
        densities = sorted(
            eos_summary.loc[
                (np.isclose(eos_summary["kT"], kT))
                & eos_summary[pressure_col].between(-0.03, 0.18)
                & (~eos_summary["phase_separated"].astype(bool)),
                "target_rho",
            ].round(3).unique()
        )

        print("=" * 80)
        print("n_fcc_cells:", n_fcc_cells, "kT:", kT, "densities:", densities)

        for target_rho in densities:
            source = cavitation.get_source_randomization_result(
                n_fcc_cells=n_fcc_cells,
                target_rho=target_rho,
                kT=kT,
                source_nsteps=source_nsteps,
                source_seed=source_seed,
                create_source_if_missing=True,
            )

            if source["frame"] is None:
                print("missing source:", n_fcc_cells, target_rho, kT)
                continue

            phase = cavitation._source_phase_separation(source)

            if phase["phase_separated"]:
                print("skipping phase-separated source:", n_fcc_cells, target_rho, kT)
                continue

            for radius in radii:
                result = cavitation.get_or_create_cavitation(
                    n_fcc_cells=n_fcc_cells,
                    target_rho=target_rho,
                    kT=kT,
                    source_nsteps=source_nsteps,
                    source_seed=source_seed,

                    radius=radius,

                    evolve_nsteps=evolve_nsteps,
                    evolve_seed=evolve_seed,

                    log_period=1_000,
                    trajectory_period=1_000,

                    random_location=False,

                    overwrite=False,
                    overwrite_initial=False,
                    overwrite_source=False,
                    create_source_if_missing=False,
                    reject_phase_separated_source=True,
                )

                print(
                    "cavitation:",
                    "n=", n_fcc_cells,
                    "rho=", target_rho,
                    "kT=", kT,
                    "radius=", radius,
                    "status=", result["status"],
                )

EOS sweep: n=30 kT=0.700 rho=0.710 direction=up
Loaded existing FCC lattice:
/exp/e961/data/MDsims-data/pnichols/Simple_Lattices_v3/FCC/n_cells_30/rho_0.710/lattice.gsd
Using GPU device
Final device: <hoomd.device.GPU object at 0x7fb7480ff8c0>
Started HDF5 logger
Log file: /exp/e961/data/MDsims-data/pnichols/Thermalized_States_v3/FCC/n_cells_30/rho_0.710/kT_0.700/nsteps_1000000/seed_1/randomization_log.hdf5
Log period: 1000


KeyboardInterrupt: 